# Prototype 24 — CLM-8B vs Jev on the rented GPU (#296 Phase 0, classifiers first)

**Question:** can the self-hosted decision model (CLM-v0.1-8B via `clm-serve`) stand in for Jev on the two jobs Jev does today?
1. **Suggest ranking** — the 10 weather/occasion scenarios from nb20, one Noul per saved outfit.
2. **Collection assignment** — the 4 target collections from nb23, one Noul per outfit.

Both models get **exactly the same state and questions**, built the way `src/ai/suggest.ts` builds them. Jev is the reference, not ground truth: the HTML report puts both top-3s side by side with your real item photos so you can judge which is right.

**Gates** (section 11):
- Suggest: CLM top-1 ∈ Jev top-3 on ≥ 75% of scenarios, median Spearman ≥ 0.6.
- Collections: CLM and Jev make the same accept/reject call (Noul ≥ 0.5) on ≥ 80% of outfit×collection pairs.
- Speed: warm CLM batch call p95 < 1 s; 4–5 parallel clients still finish without errors.
- Your eyeball check of `outputs/24/report.html`.

**How to run on rental day**
1. On the GPU box, start CLM with the repo's own scripts (**RunPod pods can't run Docker**; the reference script's flags must match how CLM was trained):
   ```
   git clone https://github.com/Contrastive-LM/CLM.git && cd CLM && pip install -e .
   GPU=0 PORT=8090 UTIL=0.35 ./serve_qwen3_8b.sh &      # encoder; log in ../logs/vllm_demo_8b.log
   CLM_API_KEY=<pick one> clm-serve --port 8700 --no-ui # downloads the ~75 MB head on first start
   ```
   Wait for `[clm] embedder … up` in clm-serve's output. On a machine with Docker, `./install.sh --gpu` starts the whole stack instead.
2. On this laptop, forward the port: `ssh -N -L 8700:localhost:8700 root@<pod-host> -p <pod-ssh-port>`
   (clm-serve isn't published on the box; the tunnel keeps it private).
3. In `.env`: `CLM_URL=http://localhost:8700/v1/systemone` and `CLM_API_KEY=<the key you picked>`. Jev uses `OPENROUTER_API_KEY` as in nb20.
4. Run all cells. Optional section 12 makes a few Qwen-Image try-ons the same way (`IMAGE_URL`, `IMAGE_API_KEY`).

Without `CLM_URL` the CLM columns are skipped, so you can dry-run everything today with Jev only.

## 1. Setup — imports, env, paths

In [ ]:
from __future__ import annotations

import base64
import io
import json
import os
import sqlite3
import statistics
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()

# Load .env if present (same convention as nb20/nb23)
env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

CLOSET_DATA_DIR = Path(os.environ.get("CLOSET_DATA_DIR", str(REPO_ROOT / ".closet")))
if not (CLOSET_DATA_DIR / "closet.db").exists():
    CLOSET_DATA_DIR = Path.home() / ".closet"
CLOSET_DB = CLOSET_DATA_DIR / "closet.db"
OUT_DIR = Path(os.environ.get("NB24_OUT_DIR", str(REPO_ROOT / "prototypes" / "outputs" / "24")))
OUT_DIR.mkdir(parents=True, exist_ok=True)

JEV_URL = os.environ.get("JEV_URL", "https://openrouter.ai/api/alpha/decisions")
JEV_KEY = os.environ.get("OPENROUTER_API_KEY", "")
JEV_MODEL = "typesafe/jev-1.13"
CLM_URL = os.environ.get("CLM_URL", "")
CLM_KEY = os.environ.get("CLM_API_KEY", "")

print(f"CLOSET_DB: {CLOSET_DB}  exists={CLOSET_DB.exists()}")
print(f"OUT_DIR:   {OUT_DIR}")
print(f"Jev:       {'key set' if JEV_KEY else 'NO OPENROUTER_API_KEY — Jev skipped'}")
print(f"CLM:       {CLM_URL or 'CLM_URL not set — CLM skipped (dry run)'}")

## 2. One decisions client for both models

Both speak TypeSafe: `POST {state, questions}` → `answers[qid]`. Two differences handled here:
- Jev needs `model` + `provider: {zdr: true}` (same as production); CLM takes neither.
- `clm-serve` answers carry **no `type`** field (`{noul: 0.41}`), so answers are read by the question asked.

In [ ]:
import urllib.request
import urllib.error


def call_decisions(url: str, key: str, state, questions: dict, *, extra: dict | None = None, timeout_s: int = 60) -> dict:
    """Returns {ok, latency_s, answers, usage, error?}."""
    if not url:
        return {"ok": False, "error": "no url", "latency_s": 0.0, "answers": {}, "usage": {}}
    body = json.dumps({**(extra or {}), "state": state, "questions": questions}).encode()
    headers = {"Content-Type": "application/json"}
    if key:
        headers["Authorization"] = f"Bearer {key}"
    req = urllib.request.Request(url, data=body, method="POST", headers=headers)
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            payload = json.loads(resp.read())
        return {"ok": True, "latency_s": time.perf_counter() - t0,
                "answers": payload.get("answers") or {}, "usage": payload.get("usage") or {}}
    except urllib.error.HTTPError as e:
        return {"ok": False, "latency_s": time.perf_counter() - t0, "answers": {}, "usage": {},
                "error": f"HTTP {e.code}: {e.read()[:300].decode(errors='replace')}"}
    except Exception as e:
        return {"ok": False, "latency_s": time.perf_counter() - t0, "answers": {}, "usage": {},
                "error": f"{type(e).__name__}: {e}"}


MODELS = {}
if JEV_KEY:
    MODELS["jev"] = lambda state, qs: call_decisions(JEV_URL, JEV_KEY, state, qs, extra={"model": JEV_MODEL, "provider": {"zdr": True}})
if CLM_URL:
    MODELS["clm"] = lambda state, qs: call_decisions(CLM_URL, CLM_KEY, state, qs)


def noul_scores(result: dict) -> dict[str, float]:
    """{qid: noul} from either model's answers."""
    out = {}
    for qid, a in (result.get("answers") or {}).items():
        if isinstance(a, dict) and isinstance(a.get("noul"), (int, float)):
            out[qid] = float(a["noul"])
    return out


# Probe each model once
probe_q = {"is_cold": {"type": "noul", "instructions": "Is the temperature below 15°C?"}}
for name, call in MODELS.items():
    r = call("It is 5°C and raining.", probe_q)
    print(f"{name:4s} ok={r['ok']}  noul={noul_scores(r)}  {r['latency_s']:.2f}s  {r.get('error') or ''}")

## 3. Data loaders — items, outfits (from nb20), photos (from nb23)

In [ ]:
def pick_active_user(db_path: Path) -> str | None:
    if not db_path.exists():
        return None
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    row = conn.execute("SELECT user_id, COUNT(*) c FROM items GROUP BY user_id ORDER BY c DESC LIMIT 1").fetchone()
    conn.close()
    return row[0] if row else None


def _rows(db_path: Path, sql: str, args: tuple) -> list[dict]:
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    rows = [dict(r) for r in conn.execute(sql, args).fetchall()]
    conn.close()
    return rows


def _json_fields(d: dict, fields: tuple[str, ...]) -> dict:
    for f in fields:
        try:
            d[f] = json.loads(d[f]) if d[f] else []
        except (json.JSONDecodeError, TypeError):
            d[f] = []
    return d


ACTIVE_USER = pick_active_user(CLOSET_DB)
items = [_json_fields(r, ("colors", "season", "occasion", "tags")) for r in _rows(CLOSET_DB, """
    SELECT id, category, subcategory, primary_color, colors, brand, material, season,
           occasion, tags, ai_description, image_uri, wear_context
    FROM items WHERE user_id = ? ORDER BY id""", (ACTIVE_USER,))] if ACTIVE_USER else []
outfits = [_json_fields(r, ("item_ids", "season", "tags")) for r in _rows(CLOSET_DB, """
    SELECT id, name, item_ids, occasion, season, tags FROM outfits WHERE user_id = ? ORDER BY id""", (ACTIVE_USER,))] if ACTIVE_USER else []
items_by_id = {it["id"]: it for it in items}
outfits = [o for o in outfits if any(i in items_by_id for i in o["item_ids"])]
print(f"Active user: {ACTIVE_USER}   items: {len(items)}   outfits with items: {len(outfits)}")
assert len(outfits) >= 2, "Need at least 2 saved outfits (npm run seed:dev, or save some in the PWA)."

try:
    from PIL import Image
    HAVE_PIL = True
except ImportError:
    HAVE_PIL = False
_IMG_CACHE: dict[str, str] = {}


def image_data_url(image_uri: str | None, *, max_dim: int = 140) -> str:
    """Inline thumbnail. Photos live under users/<id>/images/…, or images/… for the legacy 'self' user."""
    if not image_uri:
        return ""
    if image_uri in _IMG_CACHE:
        return _IMG_CACHE[image_uri]
    path = next((p for p in (CLOSET_DATA_DIR / "users" / str(ACTIVE_USER) / image_uri, CLOSET_DATA_DIR / image_uri) if p.exists()), None)
    if path is None:
        _IMG_CACHE[image_uri] = ""
        return ""
    raw = path.read_bytes()
    if HAVE_PIL:
        try:
            img = Image.open(io.BytesIO(raw))
            img.thumbnail((max_dim, max_dim))
            buf = io.BytesIO()
            img.convert("RGB").save(buf, format="JPEG", quality=75)
            raw = buf.getvalue()
        except Exception:
            pass
    _IMG_CACHE[image_uri] = "data:image/jpeg;base64," + base64.b64encode(raw).decode()
    return _IMG_CACHE[image_uri]


def outfit_image_html(outfit: dict, *, max_items: int = 6, size: int = 56) -> str:
    cells = []
    for iid in outfit.get("item_ids", [])[:max_items]:
        url = image_data_url((items_by_id.get(iid) or {}).get("image_uri"))
        if url:
            cells.append(f"<img src='{url}' style='height:{size}px;width:{size}px;object-fit:cover;border:1px solid #ddd;border-radius:6px;margin:2px'/>")
    return "<div style='display:flex;flex-wrap:wrap'>" + "".join(cells) + "</div>"


display(HTML(f"<b>{outfits[0]['name']}</b>" + outfit_image_html(outfits[0])))

## 4. Questions — built like production

`src/ai/decisionText.ts` + `suggestOutfitQuestion` in `src/ai/suggest.ts`: one Noul per outfit, instruction text + the outfit's JSON (item fields, description capped at 200 chars). Both models get these exact strings.

CLM reads at most **2048 tokens** per text (`clm-serve --max-tokens`). Vastra shrinks longer questions before sending (#351); here we just show the longest one so you can see if any outfit is near the limit (~3 characters per token).

In [ ]:
def item_blob(it: dict) -> dict:
    desc = (it.get("ai_description") or "").strip()
    blob = {
        "category": it.get("category"), "subcategory": it.get("subcategory"), "primaryColor": it.get("primary_color"),
        "colors": it.get("colors"), "material": it.get("material"), "brand": it.get("brand"),
        "season": it.get("season"), "occasion": it.get("occasion"), "tags": it.get("tags"),
        "wearContext": it.get("wear_context"), "description": (desc[:199] + "…") if len(desc) > 200 else (desc or None),
    }
    return {k: v for k, v in blob.items() if v not in (None, [], "")}


def outfit_blob(o: dict) -> dict:
    head = {"name": o.get("name"), "occasion": o.get("occasion"), "season": o.get("season"), "tags": o.get("tags")}
    head = {k: v for k, v in head.items() if v not in (None, [], "")}
    return {**head, "items": [item_blob(items_by_id[i]) for i in o["item_ids"] if i in items_by_id]}


def suggest_question(o: dict) -> dict:
    return {"type": "noul", "instructions": f"Is this outfit a good pick for the given context? Outfit JSON: {json.dumps(outfit_blob(o))}"}


SUGGEST_QUESTIONS = {f"q_{o['id']}": suggest_question(o) for o in outfits}
longest = max(len(q["instructions"]) for q in SUGGEST_QUESTIONS.values())
print(f"{len(SUGGEST_QUESTIONS)} questions; longest {longest} chars ≈ {longest // 3} tokens (limit 2048)")

## 5. Scenarios — nb20's 10 suggest fixtures, nb23's 4 collections

In [ ]:
FIXTURES = [
    {"id": "rainy_work_morning", "weather": {"temperature": 13, "condition": "Rainy", "precipitation": 5.2, "windSpeed": 15}, "occasion": "office", "theme": None, "timeOfDay": "morning", "dayOfWeek": "Tuesday"},
    {"id": "hot_brunch_weekend", "weather": {"temperature": 32, "condition": "Sunny", "precipitation": 0, "windSpeed": 5}, "occasion": "brunch", "theme": "weekend, want to feel breezy", "timeOfDay": "midday", "dayOfWeek": "Saturday"},
    {"id": "cold_formal_interview", "weather": {"temperature": -2, "condition": "Snow", "precipitation": 1, "windSpeed": 20}, "occasion": "job interview", "theme": None, "timeOfDay": "morning", "dayOfWeek": "Monday"},
    {"id": "warm_casual_errands", "weather": {"temperature": 22, "condition": "Cloudy", "precipitation": 0, "windSpeed": 8}, "occasion": "running errands", "theme": None, "timeOfDay": "afternoon", "dayOfWeek": "Saturday"},
    {"id": "mild_evening_date", "weather": {"temperature": 17, "condition": "Clear", "precipitation": 0, "windSpeed": 3}, "occasion": "dinner date", "theme": "want to look put-together but not overdressed", "timeOfDay": "evening", "dayOfWeek": "Friday"},
    {"id": "cold_wfh", "weather": {"temperature": 4, "condition": "Cloudy", "precipitation": 0, "windSpeed": 12}, "occasion": "working from home", "theme": "comfortable but presentable for video calls", "timeOfDay": "morning", "dayOfWeek": "Wednesday"},
    {"id": "humid_wedding", "weather": {"temperature": 28, "condition": "Humid", "precipitation": 0, "windSpeed": 4}, "occasion": "outdoor wedding", "theme": None, "timeOfDay": "afternoon", "dayOfWeek": "Sunday"},
    {"id": "windy_outdoor_walk", "weather": {"temperature": 10, "condition": "Windy", "precipitation": 0, "windSpeed": 35}, "occasion": "long walk in the park", "theme": None, "timeOfDay": "midday", "dayOfWeek": "Sunday"},
    {"id": "cool_workout", "weather": {"temperature": 15, "condition": "Clear", "precipitation": 0, "windSpeed": 6}, "occasion": "outdoor run", "theme": None, "timeOfDay": "morning", "dayOfWeek": "Thursday"},
    {"id": "cold_evening_out", "weather": {"temperature": 6, "condition": "Clear", "precipitation": 0, "windSpeed": 10}, "occasion": "concert", "theme": "want to stand out a bit", "timeOfDay": "evening", "dayOfWeek": "Friday"},
]


def fixture_state(fx: dict) -> dict:
    """Same shape as buildJevState in src/ai/suggest.ts."""
    w = fx["weather"]
    return {
        "weather": {"temperature_c": w["temperature"], "condition": w["condition"], "precipitation_mm": w.get("precipitation", 0), "wind_kmh": w.get("windSpeed", 0)},
        "occasion": fx.get("occasion"), "time_of_day": fx.get("timeOfDay"), "day_of_week": fx.get("dayOfWeek"), "theme": fx.get("theme"),
    }


TARGET_COLLECTIONS = [
    {
        "key": "workwear_rotation",
        "name": "Workwear Rotation",
        "description": (
            "Office-appropriate outfits for a professional workplace. Polished, put-together, "
            "tailored pieces. Includes blazers, dress shirts, structured trousers, and dress shoes. "
            "Excludes athleisure, streetwear, and weekend-casual looks."
        ),
    },
    {
        "key": "cold_weather",
        "name": "Cold Weather",
        "description": (
            "Outfits that keep you warm in winter or late fall — heavy coats, wool layers, sweaters, "
            "thermal fabrics, boots. Excludes lightweight or summer-only pieces."
        ),
    },
    {
        "key": "weekend_casual",
        "name": "Weekend Casual",
        "description": (
            "Relaxed outfits for weekends and errands. Comfortable, low-effort, includes jeans, "
            "t-shirts, hoodies, sneakers. Excludes formal, business, or dressy pieces."
        ),
    },
    {
        "key": "outdoor_rugged",
        "name": "Outdoor Rugged",
        "description": (
            "Outfits built for the outdoors — hiking, walking, active use. Utility jackets, "
            "work boots, sturdy fabrics, layered warmth. Excludes formal, dressy, or delicate pieces."
        ),
    },
]


pd.DataFrame([{k: fx[k] for k in ("id", "occasion", "timeOfDay", "dayOfWeek")} | {"temp_c": fx["weather"]["temperature"]} for fx in FIXTURES])

## 6. Run both models on the same inputs

In [ ]:
suggest_rows = []   # one row per (model, scenario)
for fx in FIXTURES:
    state = fixture_state(fx)
    for name, call in MODELS.items():
        r = call(state, SUGGEST_QUESTIONS)
        scores = noul_scores(r)
        suggest_rows.append({"model": name, "fixture_id": fx["id"], "ok": r["ok"], "latency_s": r["latency_s"],
                             "scores": scores, "error": r.get("error")})
        top = sorted(scores, key=scores.get, reverse=True)[:3]
        print(f"[{name:3s}] {fx['id']:24s} {r['latency_s']:.2f}s  top3={[q[2:10] for q in top]}  {r.get('error') or ''}")

collection_rows = []  # one row per (model, collection, outfit)
for coll in TARGET_COLLECTIONS:
    state = f"Collection: {coll['name']}\nDescription: {coll['description']}\n"
    qs = {f"q_{o['id']}": {"type": "noul", "instructions": f"Does this outfit belong in this collection? Outfit JSON: {json.dumps(outfit_blob(o))}"} for o in outfits}
    for name, call in MODELS.items():
        r = call(state, qs)
        for qid, s in noul_scores(r).items():
            collection_rows.append({"model": name, "collection": coll["key"], "outfit_id": qid[2:], "noul": s})
        print(f"[{name:3s}] {coll['key']:24s} {r['latency_s']:.2f}s  answers={len(noul_scores(r))}  {r.get('error') or ''}")

## 7. Metrics — suggest agreement

Per scenario: does CLM's #1 appear in Jev's top-3, and how similar are the full orders (Spearman)?

In [ ]:
by = {(r["model"], r["fixture_id"]): r for r in suggest_rows}
agree_rows = []
for fx in FIXTURES:
    j, c = by.get(("jev", fx["id"])), by.get(("clm", fx["id"]))
    if not (j and c and j["scores"] and c["scores"]):
        continue
    jt = sorted(j["scores"], key=j["scores"].get, reverse=True)
    ct = sorted(c["scores"], key=c["scores"].get, reverse=True)
    shared = [q for q in j["scores"] if q in c["scores"]]
    rho = pd.Series([j["scores"][q] for q in shared]).corr(pd.Series([c["scores"][q] for q in shared]), method="spearman") if len(shared) > 1 else float("nan")
    agree_rows.append({"fixture_id": fx["id"], "clm_top1_in_jev_top3": ct[0] in jt[:3], "same_top1": ct[0] == jt[0],
                       "spearman": rho, "jev_latency_s": j["latency_s"], "clm_latency_s": c["latency_s"]})
agree_df = pd.DataFrame(agree_rows)
if agree_df.empty:
    print("Needs both models (set CLM_URL and OPENROUTER_API_KEY).")
else:
    print(f"CLM top-1 in Jev top-3: {agree_df['clm_top1_in_jev_top3'].mean():.0%}   same top-1: {agree_df['same_top1'].mean():.0%}   median Spearman: {agree_df['spearman'].median():.2f}")
agree_df

## 8. Metrics — collection agreement (accept = Noul ≥ 0.5)

In [ ]:
THRESHOLD = 0.5
coll_df = pd.DataFrame(collection_rows)
coll_summary = pd.DataFrame()
if not coll_df.empty and {"jev", "clm"} <= set(coll_df["model"]):
    wide = coll_df.pivot_table(index=["collection", "outfit_id"], columns="model", values="noul").dropna()
    wide["same_call"] = (wide["jev"] >= THRESHOLD) == (wide["clm"] >= THRESHOLD)
    coll_summary = wide.groupby("collection").agg(
        pairs=("same_call", "size"), same_call=("same_call", "mean"),
        jev_accepts=("jev", lambda s: int((s >= THRESHOLD).sum())), clm_accepts=("clm", lambda s: int((s >= THRESHOLD).sum())),
        spearman=("jev", lambda s: s.corr(wide.loc[s.index, "clm"], method="spearman")),
    )
    print(f"Same accept/reject call overall: {wide['same_call'].mean():.0%} of {len(wide)} pairs")
coll_summary

## 9. Speed — warm latency and parallel clients

One suggest-sized batch (all your outfits) sent 20 times, then with 4 and 5 clients at once. Run it while nothing else is using the GPU, then (optional) again while Qwen-Image is generating (section 12) to see the slowdown.

**GPU memory:** on the box run `nvidia-smi --query-gpu=memory.used,memory.total --format=csv` and type the numbers into `GPU_NOTES` below so they land in the report.

In [ ]:
GPU_NOTES = {"card": "", "clm_memory_used_mb": "", "notes": ""}   # fill in by hand on rental day

speed_rows = []
if "clm" in MODELS:
    state = fixture_state(FIXTURES[0])
    MODELS["clm"](state, SUGGEST_QUESTIONS)  # warm-up
    for clients in (1, 4, 5):
        runs = 20 if clients == 1 else clients * 4
        with ThreadPoolExecutor(max_workers=clients) as pool:
            results = list(pool.map(lambda _: MODELS["clm"](state, SUGGEST_QUESTIONS), range(runs)))
        lat = sorted(r["latency_s"] for r in results if r["ok"])
        p = lambda q: lat[min(len(lat) - 1, max(0, int(round(q * len(lat) + 0.5)) - 1))] if lat else float("nan")
        speed_rows.append({"clients": clients, "calls": runs, "errors": sum(not r["ok"] for r in results),
                           "p50_s": p(0.5), "p95_s": p(0.95), "questions_per_call": len(SUGGEST_QUESTIONS)})
speed_df = pd.DataFrame(speed_rows)
speed_df

## 10. Visual report — Jev's and CLM's top 3 side by side, with your photos

In [ ]:
outfits_by_id = {o["id"]: o for o in outfits}


def top3_html(model: str, fx_id: str) -> str:
    r = by.get((model, fx_id))
    if not r or not r["scores"]:
        return f"<div style='color:#999'>{model}: {(r or {}).get('error') or 'not run'}</div>"
    parts = [f"<div style='font-weight:600;margin-bottom:4px'>{model.upper()} · {r['latency_s']:.2f}s</div>"]
    for rank, qid in enumerate(sorted(r["scores"], key=r["scores"].get, reverse=True)[:3], 1):
        o = outfits_by_id.get(qid[2:])
        if not o:
            continue
        s = r["scores"][qid]
        parts.append(f"<div style='border:1px solid #eee;border-radius:6px;padding:4px;margin:4px 0'>"
                     f"<div style='font-size:12px'>#{rank} <b>{o['name'] or 'Unnamed'}</b> <span style='font-family:monospace;color:#0a7f42'>{s:.3f}</span></div>"
                     f"{outfit_image_html(o, size=48)}</div>")
    return "".join(parts)


html = ["<h2>Prototype 24 — CLM-8B vs Jev</h2>",
        f"<div style='color:#666'>Closet: {len(outfits)} outfits, {len(items)} items. Models: {', '.join(MODELS) or 'none'}. GPU: {GPU_NOTES}</div>"]
if not agree_df.empty:
    html.append(f"<p><b>Suggest:</b> CLM top-1 in Jev top-3 {agree_df['clm_top1_in_jev_top3'].mean():.0%} · same top-1 {agree_df['same_top1'].mean():.0%} · median Spearman {agree_df['spearman'].median():.2f}</p>")
if not coll_summary.empty:
    html.append("<p><b>Collections</b></p>" + coll_summary.to_html(float_format=lambda x: f"{x:.2f}"))
if not speed_df.empty:
    html.append("<p><b>CLM speed</b></p>" + speed_df.to_html(index=False, float_format=lambda x: f"{x:.3f}"))
for fx in FIXTURES:
    w = fx["weather"]
    html.append(f"<h3 style='margin-top:24px'>{fx['id']}</h3><div style='color:#666;font-size:13px'>{w['temperature']}°C {w['condition']} · {fx['occasion']} · {fx['timeOfDay']} {fx['dayOfWeek']}{' · ' + fx['theme'] if fx.get('theme') else ''}</div>")
    html.append(f"<div style='display:grid;grid-template-columns:1fr 1fr;gap:16px'><div>{top3_html('jev', fx['id'])}</div><div>{top3_html('clm', fx['id'])}</div></div>")
full_html = "".join(html)
display(HTML(full_html))

## 11. Save artefacts + gates

- `outputs/24/report.html` — self-contained side-by-side review (photos embedded).
- `outputs/24/suggest.csv`, `collections.csv`, `speed.csv`.

In [ ]:
(OUT_DIR / "report.html").write_text("<!doctype html><html><head><meta charset='utf-8'><title>Prototype 24</title>"
    "<style>body{font-family:system-ui,-apple-system,sans-serif;padding:24px;max-width:1100px;margin:auto}table{border-collapse:collapse}td,th{padding:4px 8px;border:1px solid #eee}</style>"
    f"</head><body>{full_html}</body></html>")
pd.DataFrame([{k: v for k, v in r.items() if k != "scores"} | {"scores": json.dumps(r["scores"])} for r in suggest_rows]).to_csv(OUT_DIR / "suggest.csv", index=False)
pd.DataFrame(collection_rows).to_csv(OUT_DIR / "collections.csv", index=False)
speed_df.to_csv(OUT_DIR / "speed.csv", index=False)
print(f"Wrote {OUT_DIR / 'report.html'}")

print("\n── Gates ──")
if agree_df.empty:
    print("suggest:     not run (needs both models)")
else:
    print(f"suggest top-1 ∈ top-3 ≥ 75%:   {agree_df['clm_top1_in_jev_top3'].mean():.0%}  {'PASS' if agree_df['clm_top1_in_jev_top3'].mean() >= 0.75 else 'FAIL'}")
    print(f"suggest median Spearman ≥ 0.6: {agree_df['spearman'].median():.2f}  {'PASS' if agree_df['spearman'].median() >= 0.6 else 'FAIL'}")
if not coll_summary.empty:
    same = (coll_summary["same_call"] * coll_summary["pairs"]).sum() / coll_summary["pairs"].sum()
    print(f"collections same call ≥ 80%:   {same:.0%}  {'PASS' if same >= 0.8 else 'FAIL'}")
if not speed_df.empty:
    warm = speed_df[speed_df["clients"] == 1].iloc[0]
    print(f"CLM warm p95 < 1 s:            {warm['p95_s']:.2f}s  {'PASS' if warm['p95_s'] < 1 else 'FAIL'}")
    print(f"4–5 clients without errors:    {int(speed_df[speed_df['clients'] > 1]['errors'].sum())} errors  {'PASS' if speed_df[speed_df['clients'] > 1]['errors'].sum() == 0 else 'FAIL'}")
print("eyeball:       open report.html — would you wear CLM's #1?")

### Reading the results

- **Agreement is not accuracy.** Jev is the reference because it's in production, not because it's right. Where they disagree, the report decides: whose #1 would you actually wear?
- **Low Spearman but good top-1** is fine for suggest (only the top few are shown). The reverse is not.
- **Collections:** if CLM accepts far more or fewer outfits than Jev, the 0.5 threshold may need a per-model value rather than a different model.
- **Speed:** CLM is an embedding model with small heads, so it should be much faster than Jev over the network. A p95 over 1 s on a warm GPU means something's wrong in the serving setup, not the model.

## 12. (Optional) Qwen-Image — just make a few images

After the CLM run, stop CLM and start Qwen-Image on the pod (vLLM-Omni, #296 Phase 3: `vllm serve Qwen/Qwen-Image-2.1 --omni --port 8091 --api-key <key>`). Then set `IMAGE_URL=http://localhost:8091/v1` (tunnel with `ssh -L 8091:localhost:8091 …`) and `IMAGE_API_KEY`. Uses the same multipart request as Vastra (#352): garment photos first, person photo last. Put a person photo at `prototypes/inputs/person.jpg`.

In [ ]:
IMAGE_URL = os.environ.get("IMAGE_URL", "")
IMAGE_KEY = os.environ.get("IMAGE_API_KEY", "")
PERSON = Path(os.environ.get("NB24_PERSON", str(REPO_ROOT / "prototypes" / "inputs" / "person.jpg")))


def qwen_edit(prompt: str, images: list[bytes], *, steps: int = 40, timeout_s: int = 600) -> tuple[bytes | None, float, str]:
    boundary = "----nb24" + str(time.time_ns())
    parts = []
    def field(name, value):
        parts.append(f"--{boundary}\r\nContent-Disposition: form-data; name=\"{name}\"\r\n\r\n{value}\r\n".encode())
    for k, v in {"model": "Qwen/Qwen-Image-2.1", "prompt": prompt, "n": "1", "response_format": "b64_json", "output_format": "jpeg", "num_inference_steps": str(steps)}.items():
        field(k, v)
    for i, img in enumerate(images):
        parts.append(f"--{boundary}\r\nContent-Disposition: form-data; name=\"image\"; filename=\"ref-{i + 1}.jpg\"\r\nContent-Type: image/jpeg\r\n\r\n".encode() + img + b"\r\n")
    parts.append(f"--{boundary}--\r\n".encode())
    headers = {"Content-Type": f"multipart/form-data; boundary={boundary}"}
    if IMAGE_KEY:
        headers["Authorization"] = f"Bearer {IMAGE_KEY}"
    req = urllib.request.Request(f"{IMAGE_URL}/images/edits", data=b"".join(parts), method="POST", headers=headers)
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            payload = json.loads(resp.read())
        return base64.b64decode(payload["data"][0]["b64_json"]), time.perf_counter() - t0, f"peak {payload.get('metrics', {}).get('peak_memory_mb', '?')} MB"
    except Exception as e:
        return None, time.perf_counter() - t0, f"{type(e).__name__}: {e}"


def item_bytes(it: dict) -> bytes | None:
    path = next((p for p in (CLOSET_DATA_DIR / "users" / str(ACTIVE_USER) / it["image_uri"], CLOSET_DATA_DIR / it["image_uri"]) if p.exists()), None)
    return path.read_bytes() if path else None


image_html = []
if IMAGE_URL and PERSON.exists():
    for o in outfits[:3]:
        garments = [b for b in (item_bytes(items_by_id[i]) for i in o["item_ids"][:3] if i in items_by_id) if b]
        prompt = (f"Images 1–{len(garments)} are clothing items; the LAST image is the person. "
                  "Dress the person in these clothes. Keep their face, body, pose and background exactly the same.")
        out, secs, note = qwen_edit(prompt, garments + [PERSON.read_bytes()])
        print(f"{o['name']:30s} {secs:.1f}s  {note}")
        if out:
            image_html.append(f"<div><b>{o['name']}</b> · {secs:.1f}s · {note}{outfit_image_html(o, size=48)}"
                              f"<img src='data:image/jpeg;base64,{base64.b64encode(out).decode()}' style='max-width:320px;border-radius:6px'/></div>")
    if image_html:
        block = "<h2>Qwen-Image samples</h2><div style='display:grid;grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:16px'>" + "".join(image_html) + "</div>"
        display(HTML(block))
        (OUT_DIR / "report.html").write_text((OUT_DIR / "report.html").read_text().replace("</body>", block + "</body>"))
else:
    print("Skipped: set IMAGE_URL and add prototypes/inputs/person.jpg")